In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer
import xgboost as xgb

from Training_Pipeline import calc_lift

In [2]:
xgb

<module 'xgboost' from '/home/alejandro-armenta/Documents/churn/venv/lib/python3.12/site-packages/xgboost/__init__.py'>

In [3]:
grouped_data = pd.read_csv('TestChurnComplete/original.csv',index_col=[0,1])
grouped_data

y = grouped_data['is_churn'].astype(int)

X = grouped_data.drop(['is_churn'],axis=1)

#churn rate
X

,,like_per_month,newfriend_per_month,post_per_month,adview_per_month,dislike_per_month,unfriend_per_month,message_per_month,reply_per_month,account_tenure,adview_per_post,reply_per_message,like_per_post,post_per_message,unfriend_per_newfriend,dislike_pcnt,unfriend_per_newfriend_scaled,newfriend_pcnt_chng,days_since_newfriend,unfriend_28day_avg_84day_obs,unfriend_28day_avg_84day_obs_scaled
account_id,observation_date,,,,,,,,,,,,,,,,,,,,
27,2020-03-01,28.0,5.0,14.0,33.0,2.0,1.0,1.0,0.0,55.0,2.357143,0.000000,2.000000,14.000000,0.20,0.066667,0.101818,4.000000,3.0,0.333333,0.509091
51,2020-03-01,26.0,5.0,4.0,19.0,4.0,1.0,42.0,28.0,55.0,4.750000,0.666667,6.500000,0.095238,0.20,0.133333,0.203636,-0.166667,1.0,0.666667,1.018182
95,2020-03-01,74.0,5.0,64.0,17.0,20.0,0.0,12.0,6.0,55.0,0.265625,0.500000,1.156250,5.333334,0.00,0.212766,0.000000,0.250000,1.0,0.000000,0.000000
123,2020-03-01,28.0,6.0,13.0,4.0,25.0,0.0,11.0,1.0,55.0,0.307692,0.090909,2.153846,1.181818,0.00,0.471698,0.000000,1.000000,2.0,0.000000,0.000000
189,2020-03-01,3.0,1.0,8.0,16.0,2.0,0.0,0.0,0.0,55.0,2.000000,0.000000,0.375000,0.000000,0.00,0.400000,0.000000,0.000000,15.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13132,2020-05-10,19.0,0.0,20.0,2.0,3.0,0.0,27.0,6.0,25.0,0.100000,0.222222,0.950000,0.740741,0.00,0.136364,0.000000,0.000000,0.0,0.000000,0.000000
13141,2020-05-10,3.0,1.0,6.0,9.0,1.0,0.0,13.0,5.0,25.0,1.500000,0.384615,0.500000,0.461538,0.00,0.250000,0.000000,0.000000,18.0,0.000000,0.000000
13155,2020-05-10,2.0,0.0,1.0,1.0,1.0,0.0,27.0,17.0,25.0,1.000000,0.629630,2.000000,0.037037,0.00,0.333333,0.000000,0.000000,0.0,0.000000,0.000000


In [4]:
tscv = TimeSeriesSplit(
    n_splits=2)

In [5]:
score_models = {
    'lift':make_scorer(calc_lift, response_method='predict_proba'),
    'AUC':'roc_auc'
}

In [6]:
xgb_model = xgb.XGBClassifier(objective='binary:logistic')
xgb_model

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [7]:
test_params = {
    'max_depth':[1,2,4,6],
    'learning_rate':[0.1,0.2,0.3,0.4],
    'n_estimators':[20,40,80,120],
    'min_child_weight':[3,6,9,12]
}
test_params

{'max_depth': [1, 2, 4, 6],
 'learning_rate': [0.1, 0.2, 0.3, 0.4],
 'n_estimators': [20, 40, 80, 120],
 'min_child_weight': [3, 6, 9, 12]}

In [8]:
#refit es con toda la informacion

gsearch = GridSearchCV(
    estimator=xgb_model, 
    n_jobs=-1,
    scoring=score_models,
    cv=tscv,
    verbose=1,
    return_train_score=False,
    param_grid=test_params,
    refit='AUC'
)

gsearch

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [1, 2, ...], 'min_child_weight': [3, 6, ...], 'n_estimators': [20, 40, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'AUC': 'roc_auc', 'lift': make_scorer(c...redict_proba')}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'AUC'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosi

In [9]:
gsearch.fit(X,y)

Fitting 2 folds for each of 256 candidates, totalling 512 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.1, 0.2, ...], 'max_depth': [1, 2, ...], 'min_child_weight': [3, 6, ...], 'n_estimators': [20, 40, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'AUC': 'roc_auc', 'lift': make_scorer(c...redict_proba')}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'AUC'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosi

In [10]:
df = pd.DataFrame(gsearch.cv_results_)
df.sort_values(by='mean_test_AUC', ascending=False, inplace=True)

In [11]:
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,params,split0_test_lift,split1_test_lift,mean_test_lift,std_test_lift,rank_test_lift,split0_test_AUC,split1_test_AUC,mean_test_AUC,std_test_AUC,rank_test_AUC
199,0.072871,0.020963,0.019811,0.001892,0.4,1,6,120,"{'learning_rate': 0.4, 'max_depth': 1, 'min_ch...",3.458251,3.847048,3.652650,0.194399,72,0.713745,0.725503,0.719624,0.005879,1
195,0.074125,0.017736,0.018174,0.000167,0.4,1,3,120,"{'learning_rate': 0.4, 'max_depth': 1, 'min_ch...",3.351843,3.770107,3.560975,0.209132,121,0.710526,0.727320,0.718923,0.008397,2
31,0.087890,0.024354,0.019023,0.000710,0.1,2,12,120,"{'learning_rate': 0.1, 'max_depth': 2, 'min_ch...",3.671066,4.077871,3.874469,0.203402,1,0.712320,0.724778,0.718549,0.006229,3
203,0.073248,0.020860,0.017635,0.000512,0.4,1,9,120,"{'learning_rate': 0.4, 'max_depth': 1, 'min_ch...",3.458251,3.847048,3.652650,0.194399,72,0.714314,0.722764,0.718539,0.004225,4
206,0.058424,0.017609,0.021867,0.001410,0.4,1,12,80,"{'learning_rate': 0.4, 'max_depth': 1, 'min_ch...",3.511455,3.923989,3.717722,0.206267,45,0.716278,0.720411,0.718345,0.002066,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,0.146871,0.035688,0.023236,0.000629,0.4,6,12,120,"{'learning_rate': 0.4, 'max_depth': 6, 'min_ch...",2.660193,3.077639,2.868916,0.208723,249,0.648477,0.649565,0.649021,0.000544,252
241,0.098716,0.014315,0.025233,0.002925,0.4,6,3,40,"{'learning_rate': 0.4, 'max_depth': 6, 'min_ch...",2.873008,3.000698,2.936853,0.063845,244,0.630804,0.661251,0.646027,0.015223,253
243,0.224241,0.055278,0.033710,0.005023,0.4,6,3,120,"{'learning_rate': 0.4, 'max_depth': 6, 'min_ch...",2.500581,2.846816,2.673699,0.173117,256,0.635629,0.653808,0.644719,0.009089,254
246,0.139800,0.039229,0.036241,0.008679,0.4,6,6,80,"{'learning_rate': 0.4, 'max_depth': 6, 'min_ch...",2.660193,2.846816,2.753504,0.093311,255,0.644620,0.639146,0.641883,0.002737,255


In [12]:
df.to_csv('crossval_xgb.csv', index=False)

In [13]:
gsearch.best_estimator_

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [14]:
with open('xgb_model.pkl', 'wb') as fid:
    pickle.dump(gsearch.best_estimator_, fid)

In [15]:
0.712850 	

0.71285

In [16]:
0.714576

0.714576